# 19.12 完成整合後，怎樣回答「這個模型會什麼」？


一張總分單可能寫著80分，卻沒告訴你計算、聽音、看圖各會多少。多能力模型也是如此：如果照抄題很多，照抄表現很好，就可能蓋住工具流程和圖片的失敗。最終成品要交一張每項能力都有固定分母的矩陣，讓你知道成功在哪裡、限制在哪裡。

前置是[獨立評估](https://birdhackor.github.io/tiny-perceptron-vlm/5.10.html)、[工具選擇不同錯誤](https://birdhackor.github.io/tiny-perceptron-vlm/B.7.html)與[本章成品試用](https://birdhackor.github.io/tiny-perceptron-vlm/19.1.html)。矩陣是一張同考卷、不同階段並排的表：pretrain、SFT、joint、DPO與部署版本都做同一檢查。加入新能力後舊能力下降，就留在表裡；最後選擇哪份權重，也要基於預先說明的validation規則，而不是看test挑最漂亮的一欄。

先確認每項考題真的存在：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from collections import Counter

from tiny_perceptron.capstone import build_dataset

splits, _ = build_dataset()
counts = Counter(row["task"] for row in splits["test"])
for task, count in sorted(counts.items()):
    print("最後檢查", task, "分母", count)
print("合計", sum(counts.values()), "這只是考題數量，不是答對數量")

程式只數固定test資料，完全沒有生成回答。每個任務的count是那項分母；沒有對應任務就不能報它的成功率。正式評分另外讀取每題原始生成ID、EOS與解析結果，再按該任務規則數正確答案。平均方式也要明示：所有題直接平均會讓題多的能力佔較大比重，各任務等權平均則回答另一個問題。

最後90題已依凍結配方跑完，所有原始回答都保留。先說能力界線：推薦成品仍不會可靠辨識留出的形狀，也會讀錯計算器結果；它只學幾種合成圖形、兩群純音、窄範圍句型與一個加法工具。沒有一般照片理解、語音辨識、網路搜尋或通用安全與信心校準保證。下面每個分母都對應實際題目，而不是把教過的技術名字當成能力。

這張表的第一個分數欄對應報告的`action_correct`：模型首段必須正常產生EOS，且完整文字等於`expected_action`。TOOL包含工具名與有順序的參數，DIRECT／ASK也包含回答或求助內容，不是只選對標記。例如真值`DIRECT:square`、模型`DIRECT:circle`，雖然解析為合法direct且有EOS，完整字串仍不吻合，該欄算錯；回填題答`DIRECT:0`而真值`DIRECT:1`也是如此。

第二個分數欄`end_to_end_correct`再要求最後答案符合真值。直接回答沒有額外一次生成；工具題則要先執行，再讀回並由模型回答，所以首段請求全對仍可能整題錯。這解釋了下面80／90個首段吻合，但只78／90題完成：差兩題就是計算器得到1、模型讀回後答錯的例子。這個精確字串指標與B.7的選卡分類不是同一個分數。

| 推薦joint的最後任務 | 首段完整輸出吻合 | 整題正確 | 結果的範圍 |
| --- | --- | --- | --- |
| 計算器完整迴圈 | 12／12 | 10／12 | 兩次工具算對、模型讀回答錯 |
| 計算器不可用 | 12／12 | 12／12 | 正確ASK且未執行，固定模板 |
| 單獨回填工具結果 | 5／6 | 5／6 | 0+1回報1仍答錯 |
| 加法概念說明 | 6／6 | 6／6 | 固定「把兩個數合起來」答案 |
| 原始圖片顏色 | 9／9 | 9／9 | 原題全為green，另有換色對照 |
| 原始圖片形狀 | 0／9 | 0／9 | 綠色方形全部答circle |
| 顏色與音高聯合 | 18／18 | 18／18 | color、pitch兩部分都對，不含形狀 |
| 單獨音高 | 6／6 | 6／6 | low／high各3題，只辨純音 |
| 已提供短資料的問答 | 3／3 | 3／3 | 桌子／書櫃／抽屜各一題，無新增檢索 |
| 缺數量求助 | 3／3 | 3／3 | 固定ASK句子 |
| 密碼拒絕示範 | 3／3 | 3／3 | 本課窄範圍規則 |
| 簡短照抄 | 3／3 | 3／3 | 指定數字與短回答 |
| 全部題目直接合計 | 80／90 | 78／90 | 不是每項任務等權平均 |

拒絕題3／3之外，另外87題沒有產生「不能提供他人密碼」這個拒絕句，整題正確75／87；因此不能說模型只是對每題都拒絕。但這仍只排除本題庫的那種誤拒，沒有測完所有正常問法。圖片成對對照是27／36對都正確，其中顏色9／9、形狀0／9、聯合18／18；換聲18對都正確，詳見[19.6](https://birdhackor.github.io/tiny-perceptron-vlm/19.6.html)。只要原圖有錯，換後答對也不能把整對列為成功。

同一份最後考卷，也評了之前各站與壓縮版本。這裡的「整題」都要求正常結束、動作規格與最後內容正確，沒有把工具執行成功當成助理答對。

| 權重版本 | 最後整題正確 |
| --- | --- |
| 未訓練 | 0／90 |
| 預訓練300步後 | 0／90 |
| SFT後 | 45／90 |
| 推薦joint | 78／90 |
| DPO比較分支 | 78／90 |
| joint 4-bit／8-bit | 各78／90 |
| DPO 4-bit／8-bit | 各78／90 |
| 小Dense CE學生 | 62／90 |
| 小Dense KD學生 | 61／90 |
| 小Dense KD學生4-bit | 61／90 |

DPO在最後考卷與joint同分，不會把先前驗證退步的事抹掉，也不會改成用最後考卷重新選模型。推薦joint的決定與依據在[測試前的選擇紀錄](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/capstone-selection.json)；全版本分數在[正式部署實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/capstone_deployment.json)。推薦成品的[完整90題紀錄](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/capstone-evidence/deployment/test-joint.json)與[原始資料](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/capstone-evidence/deployment/data.json)用相同`id`連起來；其餘版本也保留在同一證據目錄。量化同分卻有少數生成改變，不能用這張總表宣稱逐token相同，詳見[19.10](https://birdhackor.github.io/tiny-perceptron-vlm/19.10.html)。

效率則是另一種證據：快取的12種預選驗證輸入原始ID一致；一個短照抄題在L4的full／cache中位時間是68.054／59.978毫秒。它沒有提高上表能力分數，也沒有量生成記憶體峰值，完整範圍見[19.9](https://birdhackor.github.io/tiny-perceptron-vlm/19.9.html)。把能力、速度與檔案大小分開說，學生才能知道下一步該增加哪種資料與檢查。

小Dense支線也已完成，但保留能力需要逐項看。它有79,920個參數，教師是事先選定的DPO分支；不是推薦joint的蒸餾版。下面每欄都用同一份最後題目，4-bit學生是儲存後重新載入的版本。

| 學生能力檢查 | CE學生 | KD學生 | KD 4-bit |
| --- | --- | --- | --- |
| 完整計算器迴圈 | 1／12 | 0／12 | 0／12 |
| 工具不可用 | 12／12 | 12／12 | 12／12 |
| 獨立回填結果 | 1／6 | 1／6 | 1／6 |
| 概念說明 | 6／6 | 6／6 | 6／6 |
| 原始圖片顏色／形狀 | 9／9、0／9 | 9／9、0／9 | 9／9、0／9 |
| 圖音聯合／單獨音高 | 18／18、6／6 | 18／18、6／6 | 18／18、6／6 |
| 短資料問答／缺數量／拒絕 | 各3／3 | 各3／3 | 各3／3 |
| 簡短照抄 | 0／3 | 0／3 | 0／3 |

學生的原圖顏色9／9同樣受常數基線限制，主joint的換圖／換聲對照沒有替學生另跑，不能貼給學生當成成對檢查已通過。這次KD比CE少對的一題涉及錯工具參數及錯讀回，4-bit同分也有6題生成改變，具體例子與檔案大小見[19.10](https://birdhackor.github.io/tiny-perceptron-vlm/19.10.html)，全部逐題檔見[學生實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/capstone_student.json)與其證據目錄。縮小確實完成，原能力卻沒有全部保留，這也是應該交給學生看的結果。

如果你想繼續擴展，先選矩陣的一項：換圖片分佈，就回到[第10章](https://birdhackor.github.io/tiny-perceptron-vlm/10.1.html)研究編碼與留出素材；想增加語句音訊，就回到[第12章](https://birdhackor.github.io/tiny-perceptron-vlm/12.1.html)保留時間資訊；想讓陌生問題更穩，就回到[B.7](https://birdhackor.github.io/tiny-perceptron-vlm/B.7.html)增加獨立問法評估。整合成品是一張可檢查的出發地圖，不是把所有技術名字打勾後的終點。

練習從表裡選一項，寫一個當前小世界以外的輸入與它需要新增的檢查。例如「真實街景裡紅色車輛」不能直接沿用合成色塊準確率；必須先準備有來源和標註的照片、素材家族切分與新評估。先說清楚要證明什麼，再決定增加哪種模型或訓練方法。
